# Subgraphs and Ego Networks

In [ ]:
import networkx as nx
import os.path
import matplotlib.pyplot as plt
%matplotlib inline

In this notebook we will examine **subgraphs** - i.e. a subset of the nodes in a network, along with all of the edges connecting these nodes.

We will use the sample network of LinkedIn connections that we encountered in previous labs for illustrative purposes.

In [ ]:
g = nx.read_gexf("linkedin25.gexf")
# get list of node IDs
print(list(g.nodes()))

## Subgraphs

Given a network, we create a subgraph by specifying a list of nodes. The `subgraph()` method extracts these nodes along with any edges that exist between them in the original network:

In [ ]:
required = ["Alison Gayle", "David Lennon", "Claire Scott", "Joyce Walsh", "Eric Smith"]
# create the subgraph
sg = g.subgraph(required)
# check the size of the new subgraph
print(f"Subgraph has {sg.number_of_nodes()} nodes and {sg.number_of_edges()} edges")

In [ ]:
# check the nodes in the subgraph
list(sg.nodes())

In [ ]:
# check the edges in the subgraph
list(sg.edges())

The subgraph can be treated as a normal network for all analytical purposes, including visualisation, characterisation, and centrality calculations. This allows us to focus our analysis on specific subsets of nodes whilst maintaining the relational structure.

In [ ]:
plt.figure(figsize=(8, 6))
plt.margins(0.1, 0.1)
nx.draw_networkx(sg, 
                 with_labels=True, 
                 font_size=11, 
                 node_size=1100, 
                 node_color="lightblue")
plt.axis("off")
plt.show()

When we calculate network measures (e.g., density, degree centrality), these measures are calculated with respect to the subgraph, not the overall network. This means that centrality scores and other metrics reflect the node's importance within the subgraph context.

In [ ]:
print(f"Subgraph density={nx.density(sg):.3f}")

In [ ]:
deg_scores = dict(nx.degree_centrality(sg))

for node, score in deg_scores.items():
    print(f"{node} = {score:.3f}")

We can highlight a specific subgraph in the overall visualisation by using different colours or styling. This helps to identify the subgraph's position and connections in the broader network context:

In [ ]:
plt.figure(figsize=(10, 8))
plt.margins(0.1, 0.1)
# determine node positions
pos = nx.spring_layout(g)
# draw the full network
nx.draw_networkx(g, pos, 
                 with_labels=True, 
                 font_size=11, 
                 node_size=800, 
                 node_color="#FFFF66")
# draw the subgraph set of nodes
nx.draw_networkx_nodes(g, pos, 
                       nodelist=required, 
                       node_size=800, 
                       node_color="coral")
plt.axis("off")
plt.show()

## Ego Networks

An ego network is a specific type of subgraph which consists of a focal node (the **ego**) and the nodes to which the ego is directly connected (the **alters**), plus all edges amongst the alters. This structure is particularly useful for analysing the local neighbourhood around a specific node of interest.

Let's examine an example node from the LinkedIn network to understand this concept better.

In [ ]:
ego_node = "Claire Scott"
alters = g.neighbors(ego_node)
list(alters)

We create an **ego network** in NetworkX by calling `nx.ego_graph()`, specifying the full network and the ego node. This function automatically extracts the ego node, its direct neighbours, and all connections between them:

In [ ]:
eg = nx.ego_graph(g, ego_node)

In [ ]:
# check the nodes in the ego network
list(eg.nodes())

In [ ]:
# check the edges in the ego network
list(eg.edges())

We can then create a customised visualisation of this ego network, highlighting the ego node with a different style to distinguish it from the alters:

In [ ]:
plt.figure(figsize=(9, 7))
plt.margins(0.1, 0.1)
# position all nodes
pos = nx.spring_layout(g)
# draw the full network
nx.draw_networkx(eg, pos, 
                 with_labels=True, 
                 font_size=12, 
                 node_size=900, 
                 node_color="lightblue")
# draw the ego in red, with larger node size
nx.draw_networkx_nodes(eg, pos, 
                       nodelist=[ego_node], 
                       node_size=2500, 
                       node_color="coral")
plt.axis("off")
plt.show()

Let's wrap this visualisation step up as a single function:

In [ ]:
def display_ego(g, ego_node):
    # build the ego network
    eg = nx.ego_graph(g, ego_node)
    # create the figure
    plt.figure(figsize=(9, 7))
    plt.margins(0.1, 0.1)
    title = f"Ego network for {ego_node} ({eg.number_of_nodes()} nodes, {eg.number_of_edges()} edges)"
    plt.title(title, fontsize=12)
    # lay out all nodes
    pos = nx.spring_layout(g)
    # draw the full network
    nx.draw_networkx(eg, pos, 
                     with_labels=True, 
                     font_size=12, 
                     node_size=900, 
                     node_color="lightblue")
    # draw the ego in red, with larger node size
    nx.draw_networkx_nodes(eg, pos, 
                           nodelist=[ego_node], 
                           node_size=2500, 
                           node_color="coral")
    plt.axis("off")
    plt.show()

In [ ]:
display_ego(g, "Eric Smith")

In [ ]:
display_ego(g, "Sarah Sinclair")

We could also construct the **ego minus the ego network**, which contains only the alters of the ego, excluding the ego node itself. This approach allows us to examine the connections amongst the ego's neighbours without the central node's influence:

In [ ]:
# create the full ego network
ego_node = "Sarah Sinclair"
eg = nx.ego_graph(g, ego_node)
# now remove the ego
eg.remove_node(ego_node)

We can now visualise this network, which reveals only the connections between the alters - i.e. the neighbours of the original ego node:

In [ ]:
plt.figure(figsize=(9, 7))
plt.margins(0.1, 0.1)
title = f"Ego minus ego network for {ego_node} ({eg.number_of_nodes()} nodes, {eg.number_of_edges()} edges)"
plt.title(title, fontsize=12)
# position all nodes
pos = nx.spring_layout(g)
# draw the alters
nx.draw_networkx(eg, pos, 
                 with_labels=True, 
                 font_size=12, 
                 node_size=900, 
                 node_color="lightblue")
plt.axis("off")
plt.show()